### Loading Dataset

* Downloaded [Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)
It has around 100k songs with pre-computed features like danceability, energy, tempo and etc. 
* Printed the shape of the dataset and features names

In [1]:
import pandas as pd

df = pd.read_csv('..\\archive\\dataset.csv')

print(df.shape)        
print(df.columns)     
print(df.head(3))    
print(df.info())

(114000, 21)
Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')
   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   

         album_name        track_name  popularity  duration_ms  explicit  \
0            Comedy            Comedy          73       230666     False   
1  Ghost (Acoustic)  Ghost - Acoustic          55       149610     False   
2    To Begin Again    To Begin Again          57       210826     False   

   danceability  energy  ...  loudness  mode  speechiness  acousticness  \
0         0.6

### Features Extraction and Evaluation

First, we select only numeric audio features, and drop rows with missing values 

Second, we need to normalize features to the same scale 

In [2]:
from sklearn.preprocessing import StandardScaler

feature_cols = ['danceability', 'energy', 'key','loudness',
                'speechiness', 'acousticness', 'instrumentalness', 
                'liveness', 'valence', 'tempo']

df_clean = df[feature_cols + ['track_name', 'artists']].dropna()

scaler = StandardScaler()
feature_scaled = scaler.fit_transform(df_clean[feature_cols])

print("Feature matrix shape:", feature_scaled.shape)


Feature matrix shape: (113999, 10)


### Content-Based recommender

We will try to find songs whose feature vectors are closes to the query song using cosine similarity. 

2 songs are similar if their features profiles point in the same direction

In [3]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

pca = PCA(n_components=10, random_state=42)
features = feature_scaled.astype(np.float32)
features_reduced = pca.fit_transform(features)
features_reduced = normalize(features_reduced, norm='l2', axis=1)

nn = NearestNeighbors(metric='euclidean', algorithm='brute')
nn.fit(features_reduced)

def get_recommendations(song_name, n=5):
    matches = df_clean[df_clean['track_name'].str.lower() == song_name.lower()].index
    if len(matches) == 0:
        return "Song not found."
    song_idx = matches[0]
    query = features[song_idx].reshape(1, -1)
    query_reduced = pca.transform(query)
    query_reduced = normalize(query_reduced, norm='l2')
    distances, indices = nn.kneighbors(query_reduced, n_neighbors=n+1)
    top_indices = indices.flatten()[1:]
    return df_clean.iloc[top_indices][['track_name', 'artists']]

print(get_recommendations("Used To", n=5))

           track_name                                    artists
73405  How Many Times                          andhim;Elderbrook
31936  Summer Feeling                         Matoma;Jonah Kagen
30836  Summer Feeling                         Matoma;Jonah Kagen
53886  Summer Feeling                         Matoma;Jonah Kagen
83262    Live On Love  Armin van Buuren;Diane Warren;My Marianne


### Evaluation

In [7]:
def precision_at_k(recommended, relevant, k=5):
    top_k = recommended[:k]
    hits = len(set(top_k) & set(relevant))
    return hits / k

# Example:
recommended = ['Radiohead', 'Portishead', 'Massive Attack', 'Bjork', 'The Cure']
relevant = ['Radiohead', 'Massive Attack', 'Sigur Ros']  # Ground truth

p_at_5 = precision_at_k(recommended, relevant, k=5)
print(f"Precision@5: {p_at_5:.2f}")  # 2 hits out of 5 → 0.40


Precision@5: 0.40
